In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.dim_customer AS
SELECT DISTINCT
    customerkey,
    COALESCE(gender,'unknown') AS gender,
    COALESCE(continent,'unknown') AS continent
FROM electronics_cat.silver.customers;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.dim_product AS
SELECT DISTINCT
    productkey,
    category,
    subcategory
FROM electronics_cat.silver.products;

CREATE OR REPLACE TABLE electronics_cat.gold.dim_store AS
SELECT DISTINCT
    store_key,
    COALESCE(country,'unknown') AS country
FROM electronics_cat.silver.stores;

CREATE OR REPLACE TABLE electronics_cat.gold.dim_date AS
SELECT DISTINCT
    order_date AS date,
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    DAY(order_date) AS day
FROM electronics_cat.silver.sales
WHERE order_date IS NOT NULL;

CREATE OR REPLACE TABLE electronics_cat.gold.dim_exchange_rate AS
SELECT
    date,
    currency,
    exchange
FROM electronics_cat.silver.exc_rate;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.fact_sales AS
SELECT
    s.order_number,
    s.line_item,

    s.order_date,
    YEAR(s.order_date) AS year,
    MONTH(s.order_date) AS month,

    s.customerkey,
    s.product_key,
    s.storekey,

    s.quantity,

    p.unit_price_usd,

    CAST(e.exchange AS DECIMAL(10,2)) AS exchange,

    -- ✅ revenue
    ROUND(s.quantity * p.unit_price_usd * e.exchange,2) AS revenue_usd,

    -- ✅ delivery days
    DATEDIFF(s.delivery_date, COALESCE(s.order_date, s.delivery_date)) AS delivery_days,

    -- 🔥 FINAL CHANNEL FIX
    CASE 
        WHEN s.storekey IS NULL OR s.storekey = 0 THEN 'online'
        ELSE 'store'
    END AS channel

FROM electronics_cat.silver.sales s
LEFT JOIN electronics_cat.silver.products p ON s.product_key = p.productkey
LEFT JOIN electronics_cat.silver.exc_rate e ON s.order_date = e.date AND s.currency_code = e.currency;